# Final Project: Image Style Transfer

This notebook keeps the entire project workflow in one place. The only files outside the notebook are the input images read from `content/` and `style/`, plus generated outputs saved to `results/`.

Project direction: use the TensorFlow Hub arbitrary image stylization model as a baseline, then test pipeline-level improvements such as style-strength control, color preservation, sharpening, and simple evaluation metrics.


## 1. Setup and Imports

Run this cell first. If a package is missing, install it in your notebook environment before continuing.


In [ ]:
import os
from pathlib import Path
from datetime import datetime

os.environ["TFHUB_MODEL_LOAD_FORMAT"] = "COMPRESSED"

import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import matplotlib.pyplot as plt
import PIL.Image
from PIL import ImageEnhance, ImageFilter

plt.rcParams["figure.figsize"] = (10, 10)
plt.rcParams["axes.grid"] = False
print("TensorFlow version:", tf.__version__)


## 2. Project Configuration

Change only these paths when testing new image pairs. Put images in the `content/` and `style/` folders.


In [ ]:
# Folders
CONTENT_DIR = Path("content")
STYLE_DIR = Path("style")
RESULTS_DIR = Path("results")
BASELINE_DIR = RESULTS_DIR / "baseline"
EXPERIMENT_DIR = RESULTS_DIR / "experiments"

for folder in [CONTENT_DIR, STYLE_DIR, RESULTS_DIR, BASELINE_DIR, EXPERIMENT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Input image paths. Replace these with your own filenames.
content_image_path = CONTENT_DIR / "content_demo_img.jpeg"
style_image_path = STYLE_DIR / "style_demo_img.jpeg"

# Image size settings
MAX_CONTENT_DIM = 512
MAX_STYLE_DIM = 512

# Output naming
RUN_NAME = datetime.now().strftime("run_%Y%m%d_%H%M%S")
print("Current run:", RUN_NAME)
print("Content path:", content_image_path)
print("Style path:", style_image_path)


## 3. Helper Functions


In [ ]:
def tensor_to_image(tensor):
    """Convert a TensorFlow image tensor in [0, 1] to a PIL image."""
    tensor = tf.clip_by_value(tensor, 0.0, 1.0)
    tensor = tensor * 255
    tensor = np.array(tensor, dtype=np.uint8)
    if tensor.ndim > 3:
        assert tensor.shape[0] == 1
        tensor = tensor[0]
    return PIL.Image.fromarray(tensor)


def load_img(path_to_img, max_dim=512):
    """Load image from disk, convert to float32, resize, and add batch dimension."""
    path_to_img = str(path_to_img)
    if not Path(path_to_img).exists():
        raise FileNotFoundError(f"Could not find image: {path_to_img}")

    img = tf.io.read_file(path_to_img)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)

    shape = tf.cast(tf.shape(img)[:-1], tf.float32)
    long_dim = tf.reduce_max(shape)
    scale = max_dim / long_dim
    new_shape = tf.cast(shape * scale, tf.int32)

    img = tf.image.resize(img, new_shape)
    return img[tf.newaxis, :]


def display_img(image, title=None):
    """Display a tensor or PIL image without axes."""
    if isinstance(image, PIL.Image.Image):
        plt.imshow(image)
    else:
        if len(image.shape) > 3:
            image = tf.squeeze(image, axis=0)
        plt.imshow(tf.clip_by_value(image, 0.0, 1.0))
    plt.axis("off")
    if title:
        plt.title(title)


def show_side_by_side(images, titles, figsize=(15, 5)):
    """Display multiple images in one row."""
    plt.figure(figsize=figsize)
    for i, (image, title) in enumerate(zip(images, titles), start=1):
        plt.subplot(1, len(images), i)
        display_img(image, title)
    plt.tight_layout()
    plt.show()


def save_tensor_image(tensor, path):
    """Save a tensor image to disk and return the output path."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    img = tensor_to_image(tensor)
    img.save(path)
    return path


## 4. Load and Preview Images


In [ ]:
content_image = load_img(content_image_path, max_dim=MAX_CONTENT_DIM)
style_image = load_img(style_image_path, max_dim=MAX_STYLE_DIM)

show_side_by_side(
    [content_image, style_image],
    ["Content Image", "Style Image"],
    figsize=(12, 6)
)

print("Content tensor shape:", content_image.shape)
print("Style tensor shape:", style_image.shape)


## 5. Load the Pre-trained Baseline Model

This is the TensorFlow Hub model used as the project baseline.


In [ ]:
MODEL_URL = "https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2"
hub_model = hub.load(MODEL_URL)
print("Loaded model:", MODEL_URL)


## 6. Baseline Style Transfer

This cell saves a baseline output so later improvements can be compared against it.


In [ ]:
def run_baseline_style_transfer(content_tensor, style_tensor):
    """Run the TensorFlow Hub arbitrary style transfer model."""
    stylized = hub_model(tf.constant(content_tensor), tf.constant(style_tensor))[0]
    return tf.clip_by_value(stylized, 0.0, 1.0)

baseline_output = run_baseline_style_transfer(content_image, style_image)

show_side_by_side(
    [content_image, style_image, baseline_output],
    ["Content", "Style", "Baseline Output"],
    figsize=(16, 6)
)

baseline_path = BASELINE_DIR / f"{RUN_NAME}_baseline.png"
save_tensor_image(baseline_output, baseline_path)
print("Saved baseline output to:", baseline_path)


## 7. Improvement Functions

These are controlled, project-friendly alterations to the current model pipeline. They do not require training a new model, but they let you test whether output quality improves compared with the baseline.

Implemented options:
- `style_strength`: blends the stylized image with the original content image. Lower values preserve content more; higher values preserve style more.
- `preserve_content_color`: transfers the original content color onto the stylized result while keeping stylized texture/detail.
- `sharpness`: applies a PIL sharpness enhancement after style transfer.


In [ ]:
def resize_to_match(source, target):
    """Resize source tensor image to match target tensor image height and width."""
    target_shape = tf.shape(target)[1:3]
    return tf.image.resize(source, target_shape)


def apply_style_strength(content_tensor, stylized_tensor, style_strength=1.0):
    """Blend stylized output with content image to control stylization intensity."""
    style_strength = float(np.clip(style_strength, 0.0, 1.0))
    content_resized = resize_to_match(content_tensor, stylized_tensor)
    blended = style_strength * stylized_tensor + (1.0 - style_strength) * content_resized
    return tf.clip_by_value(blended, 0.0, 1.0)


def preserve_content_color(content_tensor, stylized_tensor):
    """Keep stylized luminance/detail but use content image chrominance/color.

    This uses YUV color space:
    - Y comes from the stylized image
    - U and V come from the content image
    """
    content_resized = resize_to_match(content_tensor, stylized_tensor)
    stylized_yuv = tf.image.rgb_to_yuv(tf.squeeze(stylized_tensor, axis=0))
    content_yuv = tf.image.rgb_to_yuv(tf.squeeze(content_resized, axis=0))

    combined_yuv = tf.stack(
        [stylized_yuv[..., 0], content_yuv[..., 1], content_yuv[..., 2]],
        axis=-1
    )
    combined_rgb = tf.image.yuv_to_rgb(combined_yuv)
    combined_rgb = tf.clip_by_value(combined_rgb, 0.0, 1.0)
    return combined_rgb[tf.newaxis, :]


def apply_sharpness(tensor, sharpness=1.0):
    """Apply PIL sharpness enhancement. 1.0 means no change."""
    img = tensor_to_image(tensor)
    img = ImageEnhance.Sharpness(img).enhance(float(sharpness))
    arr = np.array(img).astype(np.float32) / 255.0
    return tf.convert_to_tensor(arr[tf.newaxis, :], dtype=tf.float32)


def run_improved_style_transfer(
    content_tensor,
    style_tensor,
    style_strength=1.0,
    preserve_color=False,
    sharpness=1.0
):
    """Run baseline model, then apply selected improvement settings."""
    output = run_baseline_style_transfer(content_tensor, style_tensor)
    output = apply_style_strength(content_tensor, output, style_strength=style_strength)

    if preserve_color:
        output = preserve_content_color(content_tensor, output)

    if sharpness != 1.0:
        output = apply_sharpness(output, sharpness=sharpness)

    return tf.clip_by_value(output, 0.0, 1.0)


## 8. Single Improved Output


In [ ]:
# Try changing these values for your experiments.
STYLE_STRENGTH = 0.85
PRESERVE_CONTENT_COLOR = False
SHARPNESS = 1.2

improved_output = run_improved_style_transfer(
    content_image,
    style_image,
    style_strength=STYLE_STRENGTH,
    preserve_color=PRESERVE_CONTENT_COLOR,
    sharpness=SHARPNESS
)

show_side_by_side(
    [content_image, style_image, baseline_output, improved_output],
    ["Content", "Style", "Baseline", "Improved / Modified"],
    figsize=(18, 6)
)

improved_path = EXPERIMENT_DIR / f"{RUN_NAME}_strength{STYLE_STRENGTH}_color{PRESERVE_CONTENT_COLOR}_sharp{SHARPNESS}.png"
save_tensor_image(improved_output, improved_path)
print("Saved improved output to:", improved_path)


## 9. Run a Small Experiment Grid

This creates multiple outputs using different settings. These are useful for your final report because you can compare them visually and with metrics.


In [ ]:
experiment_settings = [
    {"style_strength": 1.00, "preserve_color": False, "sharpness": 1.0},
    {"style_strength": 0.85, "preserve_color": False, "sharpness": 1.2},
    {"style_strength": 0.70, "preserve_color": False, "sharpness": 1.2},
    {"style_strength": 0.85, "preserve_color": True,  "sharpness": 1.2},
]

experiment_outputs = []
experiment_labels = []

for i, settings in enumerate(experiment_settings, start=1):
    output = run_improved_style_transfer(content_image, style_image, **settings)
    experiment_outputs.append(output)
    label = f"Exp {i}: strength={settings['style_strength']}, color={settings['preserve_color']}, sharp={settings['sharpness']}"
    experiment_labels.append(label)

    filename = (
        f"{RUN_NAME}_exp{i}_"
        f"strength{settings['style_strength']}_"
        f"color{settings['preserve_color']}_"
        f"sharp{settings['sharpness']}.png"
    )
    save_tensor_image(output, EXPERIMENT_DIR / filename)

show_side_by_side(experiment_outputs, experiment_labels, figsize=(22, 6))
print("Saved experiment outputs in:", EXPERIMENT_DIR)


## 10. Simple Evaluation Metrics

These metrics are not perfect, but they give you concrete comparison numbers for the final report.

- `content_mse`: lower means the output is more similar to the original content image.
- `content_ssim`: higher means the output preserves more structural similarity to the content image.
- `style_color_mse`: lower means the output has more similar average color statistics to the style image.

Use these together with visual comparison because style transfer quality is partly subjective.


In [ ]:
def prepare_for_metric(a, b):
    """Resize a to match b and clip both to [0, 1]."""
    a = resize_to_match(a, b)
    return tf.clip_by_value(a, 0.0, 1.0), tf.clip_by_value(b, 0.0, 1.0)


def content_mse(content_tensor, output_tensor):
    content_resized, output_tensor = prepare_for_metric(content_tensor, output_tensor)
    return float(tf.reduce_mean(tf.square(content_resized - output_tensor)).numpy())


def content_ssim(content_tensor, output_tensor):
    content_resized, output_tensor = prepare_for_metric(content_tensor, output_tensor)
    return float(tf.image.ssim(content_resized, output_tensor, max_val=1.0).numpy()[0])


def image_channel_mean_std(tensor):
    img = tf.squeeze(tensor, axis=0)
    mean = tf.reduce_mean(img, axis=[0, 1])
    std = tf.math.reduce_std(img, axis=[0, 1])
    return mean, std


def style_color_mse(style_tensor, output_tensor):
    style_resized, output_tensor = prepare_for_metric(style_tensor, output_tensor)
    style_mean, style_std = image_channel_mean_std(style_resized)
    output_mean, output_std = image_channel_mean_std(output_tensor)
    return float(tf.reduce_mean(tf.square(style_mean - output_mean)) + tf.reduce_mean(tf.square(style_std - output_std)))


def evaluate_output(name, output_tensor):
    return {
        "name": name,
        "content_mse": content_mse(content_image, output_tensor),
        "content_ssim": content_ssim(content_image, output_tensor),
        "style_color_mse": style_color_mse(style_image, output_tensor),
    }

results = [evaluate_output("baseline", baseline_output)]
for label, output in zip(experiment_labels, experiment_outputs):
    results.append(evaluate_output(label, output))

for row in results:
    print(row)


## 11. Final Notes / Report Talking Points

Use this section to write observations after running your experiments.

Suggested discussion points:
- Which setting best preserved the main content object?
- Which setting showed the strongest style match?
- Did color preservation help or hurt?
- Did sharpening improve perceived quality?
- How did results change when the content image had a busier background?
- Which metrics agreed with your visual judgment, and which did not?


In [ ]:
# Notes:
# 1.
# 2.
# 3.
